In [2]:
from models import *
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import TensorDataset, DataLoader
import time
from sklearn.metrics import roc_auc_score
batch_size = 32
device = 'mps'

In [3]:
def test_evaluate(n, file):
    models = {2: NeuralNet2, 3: NeuralNet3, 4: NeuralNet4, 5: NeuralNet5, 6: NeuralNet6}
    model = models[n]().to(device)
    model.load_state_dict(torch.load(file, map_location=device))
    model.eval()
    loss_function = nn.BCEWithLogitsLoss()
    correct = 0
    tp = tn = fp = fn = 0
    total_loss = 0.0
    total = 0
    test_inputs = np.memmap(f'test_data/inputs_{n}.dat', dtype=np.uint8, mode='r', shape=(20000, n, 128, 128, 3))
    test_targets = np.memmap(f'test_data/targets_{n}.dat', dtype=np.uint8, mode='r', shape=(20000,))
    dataset = TensorDataset(torch.from_numpy(test_inputs), torch.from_numpy(test_targets))
    test_loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    all_logits = []
    with torch.no_grad():
        for batch_inputs, batch_targets in test_loader:
            batch_inputs = batch_inputs.float().to(device) / 255.0
            batch_targets = batch_targets.float().to(device)
            inputs = [batch_inputs[:, i].permute(0, 3, 1, 2) for i in range(n)]
            logits = model(*inputs)
            all_logits.append(logits.squeeze().cpu().numpy())
            loss = loss_function(logits.squeeze(), batch_targets)     
            total_loss += loss.item() * batch_targets.size(0)
            total += batch_targets.size(0)
            predictions = logits.squeeze() > 0
            correct += (predictions == batch_targets).sum().item()
            tp += ((predictions == 1) & (batch_targets == 1)).sum().item()
            tn += ((predictions == 0) & (batch_targets == 0)).sum().item()
            fp += ((predictions == 1) & (batch_targets == 0)).sum().item()
            fn += ((predictions == 0) & (batch_targets == 1)).sum().item()
    all_logits = np.concatenate(all_logits)
    roc_auc = roc_auc_score(test_targets, all_logits)
    print("Test loss:", total_loss / total)
    print("Accuracy:", correct / total)
    print("ROC AUC:", roc_auc)
    print(f"TP: {tp}, TN: {tn}, FP: {fp}, FN: {fn}")

In [5]:
def inference_time(n, file):
    models = {2: NeuralNet2, 3: NeuralNet3, 4: NeuralNet4, 5: NeuralNet5, 6: NeuralNet6}
    model = models[n]().to(device)
    model.load_state_dict(torch.load(file, map_location=device))
    model.eval()
    test_inputs = np.memmap(f'test_data/inputs_{n}.dat', dtype=np.uint8, mode='r', shape=(20000, n, 128, 128, 3))
    inference_time = 0.0
    for i in range(10):
        input = torch.from_numpy(test_inputs[i:i+1].astype(np.float32)).to(device) / 255.0
        inputs = [input[:, j].permute(0, 3, 1, 2) for j in range(n)]
        _ = model(*inputs)
    for i in range(10, 510):
        input = torch.from_numpy(test_inputs[i:i+1].astype(np.float32)).to(device) / 255.0
        inputs = [input[:, j].permute(0, 3, 1, 2) for j in range(n)]
        torch.mps.synchronize()
        start = time.perf_counter()
        _ = model(*inputs)
        torch.mps.synchronize()
        end = time.perf_counter()
        inference_time += (end - start)
    print('Average inference time (sec):', inference_time / 500)

In [10]:
test_evaluate(2, 'model_2/epoch_15.pth')

Test loss: 0.31733130475282667
Accuracy: 0.8578
ROC AUC: 0.9366770050000001
TP: 8704, TN: 8452, FP: 1548, FN: 1296


In [11]:
test_evaluate(3, 'model_3/epoch_15.pth')

Test loss: 0.25839526494145393
Accuracy: 0.8894
ROC AUC: 0.9584540100000001
TP: 9044, TN: 8744, FP: 1256, FN: 956


In [6]:
test_evaluate(4, 'model_4/epoch_15.pth')

/var/folders/_b/yynhlh9s339c2gvwx12flcqm0000gp/T/ipykernel_11947/3128558641.py:13: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /Users/runner/work/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:219.)
  dataset = TensorDataset(torch.from_numpy(test_inputs), torch.from_numpy(test_targets))


Test loss: 0.2197304136455059
Accuracy: 0.90665
ROC AUC: 0.9714417399999998
TP: 9418, TN: 8715, FP: 1285, FN: 582


In [7]:
test_evaluate(5, 'model_5/epoch_14.pth')

Test loss: 0.19934426347613335
Accuracy: 0.9188
ROC AUC: 0.975583705
TP: 9259, TN: 9117, FP: 883, FN: 741


In [8]:
test_evaluate(6, 'model_6/epoch_14.pth')

Test loss: 0.20320491133332252
Accuracy: 0.91665
ROC AUC: 0.9747415100000001
TP: 9382, TN: 8951, FP: 1049, FN: 618


In [9]:
inference_time(2, 'model_2/epoch_15.pth')

Average inference time (sec): 0.003743914840000798


In [10]:
inference_time(3, 'model_3/epoch_15.pth')

Average inference time (sec): 0.0045470039039646505


In [11]:
inference_time(4, 'model_4/epoch_15.pth')

Average inference time (sec): 0.007224245994133525


In [12]:
inference_time(5, 'model_5/epoch_14.pth')

Average inference time (sec): 0.0074320206697302636


In [13]:
inference_time(6, 'model_6/epoch_14.pth')

Average inference time (sec): 0.00914757533595548
